In [36]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [37]:
PATH_ALB_CR = '../../kidneyData/P_ALB_CR.xpt'
PATH_BIOPRO = '../../kidneyData/P_BIOPRO.xpt'
PATH_KIQ    = '../../kidneyData/P_KIQ_U.xpt'
PATH_KIDNEY = '../../kidneyData/kidney_disease.csv'

paths = {
    'alb_cr': PATH_ALB_CR,
    'biopro': PATH_BIOPRO,
    'kiq': PATH_KIQ,
    'kidney': PATH_KIDNEY
}

for name, path in paths.items():
    print(f'{name}:', os.path.exists(path))

alb_cr: True
biopro: True
kiq: True
kidney: True


## UCI Kidney

In [38]:
kidney = pd.read_csv(PATH_KIDNEY)

print('Shape:', kidney.shape)
print('\nMissing values:')
print(kidney.isnull().sum())
kidney.head()

Shape: (400, 26)

Missing values:
id                  0
age                 9
bp                 12
sg                 47
al                 46
su                 49
rbc               152
pc                 65
pcc                 4
ba                  4
bgr                44
bu                 19
sc                 17
sod                87
pot                88
hemo               52
pcv                70
wc                105
rc                130
htn                 2
dm                  2
cad                 2
appet               1
pe                  1
ane                 1
classification      0
dtype: int64


,id,age,bp,sg,al,su,rbc,pc,pcc,ba,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,...,38,6000,NaN,no,no,no,good,no,no,ckd
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,...,31,7500,NaN,no,yes,no,poor,no,yes,ckd
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,...,35,7300,4.6,no,no,no,good,no,no,ckd


In [39]:
print("Original shape:", kidney.shape)

kidney['classification'] = kidney['classification'].astype(str).str.strip().str.lower()

kidney['ckd_label'] = kidney['classification'].apply(
    lambda x: 1 if 'ckd' in x and 'not' not in x else 0
)

kidney = kidney.drop(columns=['classification'])

if 'id' in kidney.columns:
    kidney = kidney.drop(columns=['id'])


cat_cols = kidney.select_dtypes(include=['object']).columns

for col in cat_cols:
    if kidney[col].isnull().sum() > 0:
        mode = kidney[col].mode()[0]
        kidney[col] = kidney[col].fillna(mode)



num_cols = kidney.select_dtypes(include=[np.number]).columns

for col in num_cols:
    if kidney[col].isnull().sum() > 0:
        median = kidney[col].median()
        kidney[col] = kidney[col].fillna(median)

kidney = kidney.drop_duplicates()


#summary after cleaning
print("\nFinal shape:", kidney.shape)

if 'ckd_label' in kidney.columns:
    print("\nCKD label counts:")
    print(kidney['ckd_label'].value_counts())

print("\nMissing values after cleaning:")
print(kidney.isnull().sum())

Original shape: (400, 26)

Final shape: (400, 25)

CKD label counts:
ckd_label
1    250
0    150
Name: count, dtype: int64

Missing values after cleaning:
age          0
bp           0
sg           0
al           0
su           0
rbc          0
pc           0
pcc          0
ba           0
bgr          0
bu           0
sc           0
sod          0
pot          0
hemo         0
pcv          0
wc           0
rc           0
htn          0
dm           0
cad          0
appet        0
pe           0
ane          0
ckd_label    0
dtype: int64


## NHANES Kidney

In [40]:
biopro = pd.read_sas(PATH_BIOPRO)  # Serum creatinine, BUN, albumin etc.
alb_cr = pd.read_sas(PATH_ALB_CR)  # Urine albumin/creatinine ratio
kiq    = pd.read_sas(PATH_KIQ)     # Kidney condition questionnaire

print('Shapes:')
print('  BIOPRO:', biopro.shape)
print('  ALB_CR:', alb_cr.shape)
print('  KIQ:   ', kiq.shape)

Shapes:
  BIOPRO: (10409, 41)
  ALB_CR: (13027, 8)
  KIQ:    (9232, 16)


In [41]:
nhanes_kidney = biopro.merge(alb_cr, on="SEQN", how="inner")
nhanes_kidney = nhanes_kidney.merge(kiq, on="SEQN", how="inner")

print("\nMerged NHANES:", nhanes_kidney.shape)

cols = [
    "SEQN",
    "LBXSCR",   # creatinine
    "LBXSBU",   # blood urea nitrogen
    "LBXSAL",   # albumin
    "LBXSPH",   # phosphorus
    "LBXSKSI",  # potassium
    "URXUMA",   # urine albumin
    "URXUCR",   # urine creatinine
    "URDACT"    # albumin/creatinine ratio
]

cols = [c for c in cols if c in nhanes_kidney.columns]
nhanes_kidney = nhanes_kidney[cols]

#remove rows with no data
bio_cols = [c for c in cols if c != "SEQN"]
nhanes_kidney = nhanes_kidney.dropna(subset=bio_cols, how="all")

for col in bio_cols:
    if nhanes_kidney[col].isna().sum() > 0:
        med = nhanes_kidney[col].median()
        nhanes_kidney[col] = nhanes_kidney[col].fillna(med)

if "URDACT" in nhanes_kidney.columns:
    nhanes_kidney["ckd_label"] = (nhanes_kidney["URDACT"] >= 30).astype(int)

print("\nCKD label counts:")
print(nhanes_kidney["ckd_label"].value_counts())

print("\nFinal shape:", nhanes_kidney.shape)
nhanes_kidney.head()

print("\nSummary statistics:")
print(nhanes_kidney.describe())


Merged NHANES: (8544, 63)

CKD label counts:
ckd_label
0    7290
1    1168
Name: count, dtype: int64

Final shape: (8458, 10)

Summary statistics:
                SEQN       LBXSCR       LBXSBU       LBXSAL       LBXSPH  \
count    8458.000000  8458.000000  8458.000000  8458.000000  8458.000000   
mean   117095.001655     0.904654    14.964058     4.051076     3.564448   
std      4499.838999     0.509240     5.926624     0.334185     0.512139   
min    109266.000000     0.250000     2.000000     2.100000     1.600000   
25%    113190.250000     0.720000    11.000000     3.900000     3.200000   
50%    117069.500000     0.840000    14.000000     4.100000     3.600000   
75%    121000.750000     0.990000    17.000000     4.300000     3.900000   
max    124822.000000    14.970000    79.000000     5.400000     9.600000   

           LBXSKSI        URXUMA       URXUCR        URDACT    ckd_label  
count  8458.000000   8458.000000  8458.000000   8458.000000  8458.000000  
mean      4.09010

In [42]:
output_dir = "../final_cleaned_data_files"
os.makedirs(output_dir, exist_ok=True)

kidney.to_csv(os.path.join(output_dir, "kidney_cleaned.csv"), index=False)
nhanes_kidney.to_csv(os.path.join(output_dir, "nhanes_kidney_cleaned.csv"), index=False)

print("Saved:")
print("  kidney_cleaned.csv")
print("  nhanes_kidney_cleaned.csv")

Saved:
  kidney_cleaned.csv
  nhanes_kidney_cleaned.csv
